## ML Model Training and Experiments

- Converting data for ML model Consumable
- Adding Weight & Biases for Experiment Tracking 
- Training Different Model 
- Evaluations

#### Data Conversion

In [1]:
# imports
import pandas as pd

In [2]:
df = pd.read_csv("../dataset/heart_disease_risk_2026.csv")
df.head()

,patient_id,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,1,44,Male,117,74,193,57,106,119,112,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,2,57,Male,139,94,185,69,110,35,114,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,3,29,Male,128,78,197,52,108,157,95,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,4,72,Male,132,86,197,59,104,143,92,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,5,62,Female,116,75,154,65,75,104,135,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1


In [3]:
df.columns.groupby(df.dtypes)

{int64: ['patient_id', 'age', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'fasting_blood_sugar', 'resting_heart_rate', 'max_heart_rate_achieved', 'exercise_minutes_per_week', 'daily_steps', 'has_heart_disease'], str: ['sex', 'chest_pain_type', 'smoker_status'], float64: ['hba1c', 'bmi', 'st_depression', 'alcohol_units_per_week', 'sleep_hours', 'stress_score', 'diet_quality_score'], bool: ['exercise_induced_angina', 'family_history', 'wearable_owner']}

In [4]:
NUMERICAL_FEATURE = ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'hdl', 'ldl', 'triglycerides', 'resting_heart_rate', 'max_heart_rate_achieved', 'exercise_minutes_per_week', 'daily_steps','hba1c', 'bmi', 'st_depression', 'alcohol_units_per_week', 'sleep_hours', 'stress_score', 'diet_quality_score'] 
CATEGORICAL_FEATURE = ['sex', 'chest_pain_type', 'smoker_status', 'exercise_induced_angina', 'family_history', 'wearable_owner'] 
TARGET_FEATURE = 'has_heart_disease'


In [5]:
# Normalizing the numerical feature 
df[NUMERICAL_FEATURE].head()

,age,resting_bp_systolic,resting_bp_diastolic,hdl,ldl,triglycerides,resting_heart_rate,max_heart_rate_achieved,exercise_minutes_per_week,daily_steps,hba1c,bmi,st_depression,alcohol_units_per_week,sleep_hours,stress_score,diet_quality_score
0,44,117,74,57,106,119,74,165,86,7731,5.2,26.5,0.3,2.9,5.4,19.8,62.9
1,57,139,94,69,110,35,63,150,132,2629,5.9,20.8,2.1,3.0,4.3,45.8,74.6
2,29,128,78,52,108,157,81,210,128,9290,5.5,24.5,0.8,3.5,5.1,17.7,65.7
3,72,132,86,59,104,143,93,137,18,7373,4.8,27.3,2.1,2.7,6.8,63.6,48.5
4,62,116,75,65,75,104,86,181,24,6331,6.2,26.0,1.3,3.3,8.2,58.7,47.3


In [6]:
X = df[NUMERICAL_FEATURE + CATEGORICAL_FEATURE]
y = df[TARGET_FEATURE]

In [7]:
y.head()

0    0
1    1
2    0
3    1
4    1
Name: has_heart_disease, dtype: int64

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
        [
            ('num-scaler',StandardScaler(), NUMERICAL_FEATURE),
            ('cat-encoder',OneHotEncoder(),CATEGORICAL_FEATURE)
        ]
)



In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train, y_test = train_test_split(X,y,train_size=0.8)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7200, 23), (1800, 23), (7200,), (1800,))

In [15]:
# Versioning the Datset in Registry in wandb
VD=0
RUN=f"raw:production-{VD}"
run_data = wandb.init(entity='aithreads24',
                      project='Threads',
                      name=RUN,
                      config={"Name":"Heart-Disease-Risk-2026","size":8990})


raw_dataset_artifact = wandb.Artifact(
    name="heart-disease-risk-dataset",
    type="dataset",
    description="Raw/Unprocesed Dataset for heart-disease-risk classifier",
)
raw_dataset_artifact.add_file("../dataset/heart_disease_risk_2026.csv")

logged_prep = run_data.log_artifact(raw_dataset_artifact)

run_data.link_artifact(
    artifact=logged_prep,
    target_path="wandb-registry-dataset/raw-dataset",
    aliases=["production"],
)
run_data.finish()

In [17]:
# Versioning the micro-dataset Datset in Registry in wandb
VD=0
RUN=f"micro-dataset:production-{VD}"
run_mdata = wandb.init(entity='aithreads24',
                      project='Threads',
                      name=RUN,
                      config={"Name":"Micro-Heart-Disease-Risk-2026","size":300})


raw_mdataset_artifact = wandb.Artifact(
    name="micro-heart-disease-risk-dataset",
    type="dataset",
    description="Raw/Unprocesed Dataset for heart-disease-risk classifier to use in CI for Pytest",
)
raw_mdataset_artifact.add_file("../dataset/pytest-micro-data/raw/heart_disease_risk_2026_300.csv")

logged_prep = run_mdata.log_artifact(raw_mdataset_artifact)

run_mdata.link_artifact(
    artifact=logged_prep,
    target_path="wandb-registry-dataset/micro-ci-pytest-dataset",
    aliases=["production"],
)
run_mdata.finish()

In [18]:
# Versioning the Golden Datset in Registry in wandb
VD=0
RUN=f"golden:production-{VD}"
run_gdata = wandb.init(entity='aithreads24',
                      project='Threads',
                      name=RUN,
                      config={"Name":"Golden-Heart-Disease-Risk-2026","size":8990})


raw_gdataset_artifact = wandb.Artifact(
    name="golden-heart-disease-risk-dataset",
    type="dataset",
    description="Raw/Unprocesed Golden Dataset for heart-disease-risk classifier",
)
raw_gdataset_artifact.add_file("../dataset/heart_disease_risk_2026.csv")

logged_prep = run_gdata.log_artifact(raw_dataset_artifact)

run_gdata.link_artifact(
    artifact=logged_prep,
    target_path="wandb-registry-dataset/golden-dataset",
    aliases=["production"],
)
run_gdata.finish()

wandb: WARNING Artifact "heart-disease-risk-dataset:v0" already exists with the same content. No new version will be created.


## Experimenting with Different ML Algorithms

- Random Forest Classifier
- Linear Classifier
- SVM Classifier


In [10]:
# common imports
import wandb
import wandb.sklearn 
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, recall_score
import joblib
import numpy as np

In [11]:

from sklearn.ensemble import RandomForestClassifier

V=2
RUN_NAME = f'random-forest-run-prod:{V}'

# 1. Initialize W&B Run
run_rf = wandb.init(
    entity="aithreads24",
    project="Threads",
    name=RUN_NAME,
    config={
        "algorithm": "RandomForest",
        "n_estimators": 150,
        "max_depth": 14,
        "random_state": 52,
    },
)

rforest_classifier = Pipeline(
    steps=[
        ("data-preprocessor", preprocessor),
        (
            "rforest-classifier",
            RandomForestClassifier(
                n_estimators=100, max_depth=10, random_state=42
            ),
        ),
    ]
)

rforestM = rforest_classifier.fit(X_train, y_train)

y_pred = rforestM.predict(X_test)
y_probas = rforestM.predict_proba(X_test)

class_names = [str(cls) for cls in rforestM.classes_]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_probas[:, 1])

print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC AUC: {auc:.4f}")

# scaler metrics log to wandb
wandb.log({
    "accuracy": acc,
    "precision": prec,
    "recall": recall,
    "f1_score": f1,
    "roc_auc_score": auc,
})

# charts logs of classifier
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None, 
        y_true=np.asarray(y_test), 
        preds=y_pred, 
        class_names=class_names
    ),
    "roc_curve": wandb.plot.roc_curve(
        y_true=np.asarray(y_test), 
        y_probas=y_probas, 
        labels=class_names
    ),
    "precision_recall_curve": wandb.plot.pr_curve(
        y_true=np.asarray(y_test), 
        y_probas=y_probas, 
        labels=class_names
    ),
})
#=====================================================Preprocessor======
# preprocessor versioning in registry in wandb
joblib.dump(preprocessor, "../models/preprocessor.joblib")

preprocessor_artifact = wandb.Artifact(
    name="heart-risk-preprocessor",
    type="model",
    description="ColumnTransformer for encoding and scaling tabular features",
)
preprocessor_artifact.add_file("../models/preprocessor.joblib")

logged_prep = run_rf.log_artifact(preprocessor_artifact)

run_rf.link_artifact(
    artifact=logged_prep,
    target_path="wandb-registry-model/heart-risk-processor",
    aliases=["production"],
)
#==================================================================
# random forest ml model versioing in registry 
joblib.dump(rforest_classifier, "../models/random_forest_classifier.joblib")

artifact = wandb.Artifact(
    name="heart-risk-ml",  # Best practice: use task-focused collection name
    type="model",
    description="Random Forest pipeline model with preprocessor",
)
artifact.add_file("../models/random_forest_classifier.joblib")

# 2. Log the artifact to your experiment run
logged_artifact = run_rf.log_artifact(artifact)

# 3. Link the logged artifact to your Registry Collection
run_rf.link_artifact(
    artifact=logged_artifact,
    target_path="wandb-registry-model/heart-risk",
    aliases=["production"],  # aliase to tag a model prod/stage
)
#=======================================================
# 4. Finish the run
run_rf.finish()



wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\verma\_netrc.
wandb: Currently logged in as: aithreads (aithreads24) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Accuracy: 0.8767 | Precision: 0.8741 | Recall: 0.6958 | F1: 0.7748 | ROC AUC: 0.9322


accuracy,▁
f1_score,▁
precision,▁
recall,▁
roc_auc_score,▁
accuracy,0.87667
f1_score,0.77485
precision,0.87414
recall,0.69581
roc_auc_score,0.93225


In [12]:

# from sklearn.ensemble import RandomForestClassifier


# # 1. Initialize W&B Run
# run_rf = wandb.init(
#     project="mtech-demo-experiments",
#     name="random-forest-run-v1",
#     config={
#         "algorithm": "RandomForest",
#         "n_estimators": 100,
#         "max_depth": 10,
#         "random_state": 42,
#     },
# )

# rforest_classifier = Pipeline(
#     steps=[
#         ("data-preprocessor", preprocessor),
#         (
#             "rforest-classifier",
#             RandomForestClassifier(
#                 n_estimators=100, max_depth=10, random_state=42
#             ),
#         ),
#     ]
# )

# rforestM = rforest_classifier.fit(X_train, y_train)

# y_pred = rforestM.predict(X_test)
# y_probas = rforestM.predict_proba(X_test)

# class_names = [str(cls) for cls in rforestM.classes_]

# acc = accuracy_score(y_test, y_pred)
# prec = precision_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred)
# auc = roc_auc_score(y_test, y_probas[:, 1])

# print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | ROC AUC: {auc:.4f}")

# wandb.log({
#     "accuracy": acc,
#     "precision": prec,
#     "recall": recall,
#     "f1_score": f1,
#     "roc_auc_score": auc,
# })

# # wandb.sklearn.plot_confusion_matrix(y_test, y_pred, labels=class_names)
# # wandb.sklearn.plot_roc(y_test, y_probas, labels=class_names)
# # wandb.sklearn.plot_precision_recall(y_test, y_probas, labels=class_names)

# wandb.log({
#     "confusion_matrix": wandb.plot.confusion_matrix(
#         probs=None, 
#         y_true=np.asarray(y_test), 
#         preds=y_pred, 
#         class_names=class_names
#     ),
#     "roc_curve": wandb.plot.roc_curve(
#         y_true=np.asarray(y_test), 
#         y_probas=y_probas, 
#         labels=class_names
#     ),
#     "precision_recall_curve": wandb.plot.pr_curve(
#         y_true=np.asarray(y_test), 
#         y_probas=y_probas, 
#         labels=class_names
#     ),
# })

# joblib.dump(rforest_classifier, "../models/rforest_classifier.joblib")

# artifact = wandb.Artifact(
#     name="RandomForestClassifier",
#     type="model",
#     description="Random Forest pipeline model with preprocessor",
# )
# artifact.add_file("../models/rforest_classifier.joblib")

# run_rf.log_artifact(artifact)

# wandb.finish()

## Versioning the Dataset and Model to wandb Registry and Collections

- upload a base model (ml and preprocessor) to models registry inside the collections (model, preprocessor)
    - run a training execution -> save model, preprocessor -> link the artifact or upload it to registry/collections
